# Crop Weed Recognition for Smart Farming
Custom CNN vs MobileNetV2 vs ResNet50 with Grad-CAM and SHAP


In [ ]:
# Install if needed
# !pip install shap grad-cam


In [ ]:
import os, time, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


## Dataset Configuration


In [ ]:
DATASET_DIR='/kaggle/input/mh-weed16'
IMG_SIZE=(224,224)
BATCH_SIZE=32
SEED=42


In [ ]:
# Load datasets
train_ds=tf.keras.utils.image_dataset_from_directory(DATASET_DIR, validation_split=0.30, subset='training', seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE)
temp_ds=tf.keras.utils.image_dataset_from_directory(DATASET_DIR, validation_split=0.30, subset='validation', seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE)
class_names=train_ds.class_names
pd.DataFrame({'class':class_names}).to_csv('results_class_names.csv',index=False)


In [ ]:
# Sample figure
fig=plt.figure(figsize=(8,8))
for images,labels in train_ds.take(1):
    for i in range(min(16,len(images))):
        ax=plt.subplot(4,4,i+1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.tight_layout(); plt.savefig('figure1_dataset_samples.pdf'); plt.close()


## Custom CNN


In [ ]:
def build_custom_cnn(num_classes):
    m=models.Sequential([
      layers.Rescaling(1./255,input_shape=(224,224,3)),
      layers.Conv2D(32,3,activation='relu'),layers.MaxPooling2D(),
      layers.Conv2D(64,3,activation='relu'),layers.MaxPooling2D(),
      layers.Conv2D(128,3,activation='relu'),layers.MaxPooling2D(),
      layers.Conv2D(256,3,activation='relu'),layers.MaxPooling2D(),
      layers.Flatten(),layers.Dense(256,activation='relu'),layers.Dropout(0.5),
      layers.Dense(num_classes,activation='softmax')])
    return m


## MobileNetV2 and ResNet50 sections (train, fine tune, save models)


In [ ]:
# Create reusable evaluation function
def evaluate_model(model,test_ds,name):
    y_true=[]; y_pred=[]
    for x,y in test_ds:
        p=np.argmax(model.predict(x,verbose=0),axis=1)
        y_pred.extend(p); y_true.extend(y.numpy())
    acc=accuracy_score(y_true,y_pred)
    pr,re,f1,_=precision_recall_fscore_support(y_true,y_pred,average='weighted')
    pd.DataFrame(classification_report(y_true,y_pred,output_dict=True)).T.to_csv(f'{name}_classification_report.csv')
    pd.DataFrame(confusion_matrix(y_true,y_pred)).to_csv(f'{name}_confusion_matrix.csv',index=False)
    return [name,acc,pr,re,f1]


## Grad-CAM and SHAP
Save all visualizations as PDF.
